# Order invariance meets floating point

A runnable check of one statement in Soufi, H. (2026), *Order Invariance at the Last Language-Model Boundary* ([SSRN 7539442](https://doi.org/10.2139/ssrn.7539442)):

> "vote aggregation is a fixed arithmetic rule and is order-invariant by construction"

**Result.** The statement holds for the rule in exact arithmetic. It does **not** hold for the floating-point implementation when the margin sits exactly on the consensus threshold. The 3,839 valid trials recorded in the study are not affected.

Standard library only. Python 3.10 or later. Run from the repository folder; no API key and no network access are needed.

In [1]:
import inspect, itertools, json, math, random
from collections import Counter
from fractions import Fraction

from common import (AGENTS, CONSENSUS_THRESHOLD, RESULTS_FILE, SETS_FILE,
                    WEIGHTS_2026_09_26, WEIGHTS_CURRENT, aggregate)

print(inspect.getsource(aggregate))

def aggregate(votes: list[tuple[str, str, float]], weights: dict) -> tuple[str, bool]:
    """Exact replica of AgentCoordinator.aggregate (agents_v2, 2026-09-27). votes: (agent, direction, confidence)."""
    scores = {"BUY": 0.0, "SELL": 0.0, "NEUTRAL": 0.0}
    total = 0.0
    for agent, direction, conf in votes:
        w = weights.get(agent, 0.0) * conf
        if w <= 0.0 or (agent == "regime" and direction == "NEUTRAL"):
            continue
        scores[direction] += w
        total += w
    if total == 0.0 or (scores["BUY"] == 0.0 and scores["SELL"] == 0.0):
        return "NEUTRAL", False
    win = "BUY" if scores["BUY"] >= scores["SELL"] else "SELL"
    opp = "SELL" if win == "BUY" else "BUY"
    margin = (scores[win] - scores[opp]) / total
    reached = margin >= CONSENSUS_THRESHOLD
    return (win if reached else "NEUTRAL"), reached



## 1. Why order can matter

Floating-point addition is commutative, but it is not associative. Reordering the terms of a sum can change its last bit.

In [2]:
print((0.1 + 0.2) + 0.3)
print(0.1 + (0.2 + 0.3))

0.6000000000000001
0.6


## 2. One input, 720 orders

Six votes whose margin is exactly one half, which is the threshold. The rule accepts a margin **greater than or equal to** the threshold.

In [3]:
votes = [("regime", "NEUTRAL", 0.05), ("ml_structure", "NEUTRAL", 0.9), ("correlation", "SELL", 0.1),
         ("fundamental", "BUY", 0.95), ("sentiment", "NEUTRAL", 0.3), ("research", "BUY", 0.6)]

def exact_margin(votes, weights):
    score, total = Counter(), Fraction(0)
    for agent, direction, conf in votes:
        w = Fraction(str(weights[agent])) * Fraction(str(conf))
        if w <= 0 or (agent == "regime" and direction == "NEUTRAL"):
            continue
        score[direction] += w
        total += w
    return abs(Fraction(score["BUY"]) - Fraction(score["SELL"])) / total

print("margin in exact arithmetic:", exact_margin(votes, WEIGHTS_2026_09_26))
orders = list(itertools.permutations(votes))
print("outcomes over", len(orders), "orders:", dict(Counter(aggregate(list(o), WEIGHTS_2026_09_26) for o in orders)))

margin in exact arithmetic: 1/2
outcomes over 720 orders: {('NEUTRAL', False): 360, ('BUY', True): 360}


## 3. How often, and two alternatives

Three implementations of the same rule:

* **float**: the implementation above (running `+=` sums);
* **fsum**: the same rule with `math.fsum`, which returns the correctly rounded sum whatever the order;
* **exact**: rational arithmetic on the decimal values.

200,000 random six-vote inputs with confidences on a 0.05 grid. Rounding error in a six-term sum is far below `1e-9`, so only inputs within `1e-9` of the threshold can change with order; those are re-evaluated under all 720 orders. The grid makes exact ties much more likely than continuous confidences would, so these counts show that the effect exists and say nothing about its frequency in any running system.

In [4]:
def aggregate_fsum(votes, weights):
    terms = {"BUY": [], "SELL": [], "NEUTRAL": []}
    for agent, direction, conf in votes:
        w = weights.get(agent, 0.0) * conf
        if w <= 0.0 or (agent == "regime" and direction == "NEUTRAL"):
            continue
        terms[direction].append(w)
    buy, sell = math.fsum(terms["BUY"]), math.fsum(terms["SELL"])
    total = math.fsum(w for group in terms.values() for w in group)
    if total == 0.0 or (buy == 0.0 and sell == 0.0):
        return "NEUTRAL", False
    reached = abs(buy - sell) / total >= CONSENSUS_THRESHOLD
    return (("BUY" if buy >= sell else "SELL") if reached else "NEUTRAL"), reached

def aggregate_exact(votes, weights):
    buy = sell = total = Fraction(0)
    for agent, direction, conf in votes:
        w = Fraction(str(weights.get(agent, 0.0))) * Fraction(str(conf))
        if w <= 0 or (agent == "regime" and direction == "NEUTRAL"):
            continue
        buy += w if direction == "BUY" else 0
        sell += w if direction == "SELL" else 0
        total += w
    if total == 0 or (buy == 0 and sell == 0):
        return "NEUTRAL", False
    reached = abs(buy - sell) / total >= Fraction(str(CONSENSUS_THRESHOLD))
    return (("BUY" if buy >= sell else "SELL") if reached else "NEUTRAL"), reached

def near_threshold(votes, weights):
    score, total = Counter(), 0.0
    for agent, direction, conf in votes:
        w = weights[agent] * conf
        if w > 0 and not (agent == "regime" and direction == "NEUTRAL"):
            score[direction] += w
            total += w
    return total > 0 and abs(abs(score["BUY"] - score["SELL"]) / total - CONSENSUS_THRESHOLD) < 1e-9

RULES = {"float": aggregate, "fsum": aggregate_fsum, "exact": aggregate_exact}
N, rng = 200_000, random.Random(20261126)
agents, grid = AGENTS + ["research"], [i / 20 for i in range(1, 21)]
near, depends, wrong = 0, Counter(), Counter()

for _ in range(N):
    v = [(a, rng.choice(["BUY", "SELL", "NEUTRAL"]), rng.choice(grid)) for a in agents]
    if not near_threshold(v, WEIGHTS_2026_09_26):
        continue
    near += 1
    truth = aggregate_exact(v, WEIGHTS_2026_09_26)
    perms = [list(p) for p in itertools.permutations(v)]
    for name, rule in RULES.items():
        seen = {rule(p, WEIGHTS_2026_09_26) for p in perms}
        depends[name] += len(seen) > 1
        wrong[name] += seen != {truth}

print(f"random inputs: {N:,}   within 1e-9 of the threshold: {near}")
print(f"{'rule':8}{'order-dependent':>18}{'differs from exact':>22}")
for name in RULES:
    print(f"{name:8}{depends[name]:>18}{wrong[name]:>22}")

random inputs: 200,000   within 1e-9 of the threshold: 230
rule       order-dependent    differs from exact
float                  160                   163
fsum                     0                    41
exact                    0                     0


`math.fsum` removes the dependence on order. It does not by itself guarantee the decimal answer on the threshold, because each product is already rounded before it is summed; the table's last column shows this. Exact arithmetic (or a tolerance that is fixed and documented in the rule) addresses both.

## 4. Are the study's recorded trials affected?

Every valid trial in `results_v3.jsonl` is combined with the other five votes of its evidence set and aggregated under all 720 orders, for both weight settings used in the paper.

In [5]:
sets = {s["set_id"]: s for s in json.loads(SETS_FILE.read_text(encoding="utf-8"))["sets"]}
records = [json.loads(l) for l in RESULTS_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]
valid = [r for r in records if r["status"] == "ok"]

for label, weights in (("research weight 1.0", WEIGHTS_2026_09_26), ("research weight 0.0", WEIGHTS_CURRENT)):
    cache, dependent, differs = {}, 0, 0
    for r in valid:
        key = (r["set_id"], r["direction"], r["confidence"])
        if key not in cache:
            v = [(a, x["direction"], x["confidence"]) for a, x in sets[r["set_id"]]["votes"].items()]
            v.append(("research", r["direction"], r["confidence"]))
            seen = {aggregate(list(p), weights) for p in itertools.permutations(v)}
            cache[key] = (len(seen) > 1, seen != {aggregate_exact(v, weights)})
        dependent += cache[key][0]
        differs += cache[key][1]
    print(f"{label}: {len(valid):,} trials, order-dependent: {dependent}, differing from exact: {differs}")

research weight 1.0: 3,839 trials, order-dependent: 0, differing from exact: 0
research weight 0.0: 3,839 trials, order-dependent: 0, differing from exact: 0


## Scope

* The paper's measured results concern the language-model component; the recorded trials give the same aggregation outcome under every order.
* The sentence quoted at the top is too strong as written. It should read: order-invariant **in exact arithmetic**; a floating-point implementation needs an order-independent sum and an explicit rule for values on the threshold.
* Nothing here estimates how often a running system meets the threshold exactly.